#**Import and run relevant functions**

**Run the code below by clicking on the play button to load the functions**




In [ ]:
!git clone https://github.com/WoutSin/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing.git

# @title
import os
import math
import pickle
from collections import defaultdict, Counter
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np


# ----------------------------- Data structures ----------------------------- #

Token = str
TaggedToken = Tuple[str, str]                # (token, tag)
TaggedSequence = List[TaggedToken]
CandidatesDict = Dict[int, List[str]]        # position -> candidates
UniDict = Dict[str, float]                   # token -> prob
BiDict = Dict[str, Dict[str, float]]         # left -> {right: prob}
TriDict = Dict[Tuple[str, str], Dict[str, float]]  # (t1, t2) -> {t3: prob}


# -------------------------------- Utilities -------------------------------- #

def tokenize_text(input_string: str) -> List[str]:
    """
    Input:
        input_string: runic sequence (string)
    Output:
        tokens: list of tokens in the runic sequence
    """
    return [tok for tok in input_string.split(" ") if tok]


def get_tags(runic_list: Sequence[str]) -> TaggedSequence:
    """
    Input:
        runic_list: tokenized runic sequence
    Output:
        output_list: list of tuples (token, tag)
    Tags:
        <com>: complete tokens (no missing characters)
        <inc>: incomplete tokens (some characters missing, contains '-' or '…' plus letters)
        <mis>: missing tokens (all characters missing, token is '…' or just dashes/ellipsis)
    """
    special_chars = ["-", "…"]
    output_list: TaggedSequence = []

    for item in runic_list:
        if any(ch in item for ch in special_chars) and any(ch not in special_chars for ch in item):
            output_list.append((item, "<inc>"))
        elif all(ch in special_chars for ch in item):
            output_list.append((item, "<mis>"))
        else:
            output_list.append((item, "<com>"))
    return output_list


def min_edit_distance(source: str, target: str) -> float:
    """
    Input:
        source: token with missing characters ('-' or '…')
        target: candidate token
    Output:
        distance: minimum edit distance (float)

    Same rules as the training script:
      - '-' matches any single character at zero cost (wildcard for one char).
      - '…' can match an arbitrary-length span at zero incremental cost when
        advancing in target (insertion-like).
      - Normal substitution cost = 2, insertion = 1, deletion = 1.
    """
    distance_matrix = np.zeros((len(source) + 1, len(target) + 1))

    for i in range(len(source) + 1):
        distance_matrix[i][0] = i
    for j in range(len(target) + 1):
        distance_matrix[0][j] = j

    for i in range(1, len(source) + 1):
        for j in range(1, len(target) + 1):
            if source[i - 1] == target[j - 1]:
                distance_matrix[i][j] = distance_matrix[i - 1][j - 1]
            elif source[i - 1] == "-":
                distance_matrix[i][j] = distance_matrix[i - 1][j - 1]
            elif source[i - 1] == "…":
                distance_matrix[i][j] = min(distance_matrix[i - 1][j - 1], distance_matrix[i][j - 1])
            else:
                distance_matrix[i][j] = min(
                    distance_matrix[i - 1][j - 1] + 2,
                    distance_matrix[i - 1][j] + 1,
                    distance_matrix[i][j - 1] + 1,
                )
    return float(distance_matrix[-1][-1])


# ----------------------------- Normalization ------------------------------ #

def load_normalization_mapping(json_path: str) -> Dict[str, str]:
    """
    Input:
        json_path: path to scandi_runic_all_mapping.json
    Output:
        runic_to_normalized: dict mapping each runic variant -> normalized form
    """
    import json
    with open(json_path, "r", encoding="utf-8") as f:
        mapping = json.load(f)

    runic_to_normalized: Dict[str, str] = {}
    for normalized, variants in mapping.items():
        for runic_variant in variants.keys():
            runic_to_normalized[runic_variant] = normalized
    return runic_to_normalized


def load_normalized_ngram_pickles() -> Tuple[UniDict, BiDict, TriDict]:
    """
    Input:
        None
    Output:
        normalized unigram, bigram, and trigram dictionaries
    """
    from collections import defaultdict
    with open("/content/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing/Notebook/unigram_tokens_normalized.pkl", "rb") as f_u:
        unigram_list: List[Tuple[str, float]] = pickle.load(f_u)
    with open("/content/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing/Notebook/bigram_tokens_normalized.pkl", "rb") as f_b:
        bigram_list: List[Tuple[Tuple[str, str], float]] = pickle.load(f_b)
    with open("/content/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing/Notebook/trigram_tokens_normalized.pkl", "rb") as f_t:
        trigram_list: List[Tuple[Tuple[str, str, str], float]] = pickle.load(f_t)

    unigrams: UniDict = {tok: prob for tok, prob in unigram_list}
    bigrams: BiDict = defaultdict(dict)
    for (w1, w2), prob in bigram_list:
        bigrams[w1][w2] = prob
    trigrams: TriDict = defaultdict(dict)
    for (w1, w2, w3), prob in trigram_list:
        trigrams[(w1, w2)][w3] = prob

    return unigrams, bigrams, trigrams


def normalize_token_with_map(token: Optional[str], runic_to_normalized: Dict[str, str]) -> Optional[str]:
    """
    Input:
        token: runic token or None
        runic_to_normalized: mapping dictionary
    Output:
        normalized token (or unchanged original / None)
    """
    if token is None:
        return None
    if not isinstance(token, str):
        return token
    return runic_to_normalized.get(token, token)


# ----------------------------- Model loading ------------------------------ #

def load_ngram_pickles() -> Tuple[UniDict, BiDict, TriDict, List[Tuple[str, float]]]:
    """
    Input:
        None
    Output:
        unigrams: dict token -> prob
        bigrams: dict left -> dict right -> prob
        trigrams: dict (t1, t2) -> dict t3 -> prob
        unigram_list: list of (token, prob) tuples (for convenience)
    """
    with open("/content/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing/Notebook/unigram_tokens.pkl", "rb") as f_u:
        unigram_list: List[Tuple[str, float]] = pickle.load(f_u)

    with open("/content/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing/Notebook/bigram_tokens.pkl", "rb") as f_b:
        bigram_list: List[Tuple[Tuple[str, str], float]] = pickle.load(f_b)

    with open("/content/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing/Notebook/trigram_tokens.pkl", "rb") as f_t:
        trigram_list: List[Tuple[Tuple[str, str, str], float]] = pickle.load(f_t)

    unigrams: UniDict = {}
    for tok, prob in unigram_list:
        # Keep max prob if duplicates from set/dedup in training extractor
        unigrams[tok] = max(prob, unigrams.get(tok, 0.0))

    bigrams: BiDict = defaultdict(dict)
    for (w1, w2), prob in bigram_list:
        prev = bigrams[w1].get(w2, 0.0)
        if prob > prev:
            bigrams[w1][w2] = prob

    trigrams: TriDict = defaultdict(dict)
    for (w1, w2, w3), prob in trigram_list:
        prev = trigrams[(w1, w2)].get(w3, 0.0)
        if prob > prev:
            trigrams[(w1, w2)][w3] = prob

    return unigrams, bigrams, trigrams, unigram_list


# ------------------------ Candidate generation logic ---------------------- #

def _rank_unique(cands: Iterable[Tuple[str, float]]) -> List[str]:
    """
    Input:
        cands: iterable of (candidate, probability)
    Output:
        ordered unique candidate list by descending probability
    """
    seen = set()
    ordered: List[str] = []
    for cand, prob in sorted(cands, key=lambda x: -x[1]):
        if cand not in seen:
            seen.add(cand)
            ordered.append(cand)
    return ordered


def generate_candidates_for_position(
    seq: TaggedSequence,
    pos: int,
    trigrams: TriDict,
    bigrams: BiDict,
    unigrams: UniDict,
    include_unigrams_for_inc: bool = True,
    max_unigram_pool: int = 2000,
) -> List[str]:
    """
    Input:
        seq: tagged sequence
        pos: target index
        trigrams, bigrams, unigrams: n-gram probability maps
        include_unigrams_for_inc: whether to add unigram-based candidates for <inc>
        max_unigram_pool: to cap work when filtering by edit distance
    Output:
        ordered candidate list for position 'pos' (may be empty)

    Category order:
      tri_com+com  -> tri_inc+com -> tri_inc+inc -> bi_com -> bi_inc -> (optional) unigrams
    """
    tok, tag = seq[pos]
    if tag == "<com>":
        return [tok]

    def neighbor(idx: int) -> Tuple[Optional[str], str]:
        if idx < 0 or idx >= len(seq):
            return None, "<out>"
        t, tg = seq[idx]
        if tg == "<com>":
            return t, "<com>"
        if tg == "<inc>":
            return None, "<inc>"
        if tg == "<mis>":
            return None, "<mis>"
        return None, tg

    candidates_by_category: Dict[str, List[Tuple[str, float]]] = {
        "tri_com+com": [],
        "tri_inc+com": [],
        "tri_inc+inc": [],
        "bi_com": [],
        "bi_inc": [],
        "uni": [],
    }

    # Trigram windows: [pos-2,pos-1,pos], [pos-1,pos,pos+1], [pos,pos+1,pos+2]
    windows = [
        (pos - 2, 2),
        (pos - 1, 1),
        (pos, 0),
    ]

    for w_start, target_idx in windows:
        i0, i1, i2 = w_start, w_start + 1, w_start + 2
        if i0 < 0 or i2 >= len(seq):
            continue
        neigh = [neighbor(i0), neighbor(i1), neighbor(i2)]
        abs_target = w_start + target_idx
        if abs_target != pos:
            continue

        other_positions = [0, 1, 2]
        other_positions.remove(target_idx)
        known_tokens: List[Optional[str]] = [neigh[i][0] for i in other_positions]
        kinds: List[str] = [neigh[i][1] for i in other_positions]

        # only proceed if both neighbors exist (not out-of-range) and at least some info available
        if any(k == "<out>" for k in kinds):
            continue

        # Decide trigram index key: (t1,t2)->t3, etc.
        if target_idx == 0:
            key = (known_tokens[0], known_tokens[1])  # (t2, t3)
            for (t1, t2), nexts in trigrams.items():
                if t2 == key[0]:
                    prob = nexts.get(key[1], None)
                    if prob is not None:
                        cat = (
                            "tri_inc+inc" if kinds.count("<inc>") == 2 else
                            "tri_inc+com" if "<inc>" in kinds else
                            "tri_com+com"
                        )
                        candidates_by_category[cat].append((t1, prob))
        elif target_idx == 1:
            left = known_tokens[0]
            right = known_tokens[1]
            if left is not None and right is not None:
                for cand, prob in trigrams.get((left, right), {}).items():
                    cat = (
                        "tri_inc+inc" if kinds.count("<inc>") == 2 else
                        "tri_inc+com" if "<inc>" in kinds else
                        "tri_com+com"
                    )
                    candidates_by_category[cat].append((cand, prob))
        else:  # target_idx == 2
            key = (known_tokens[0], known_tokens[1])
            if key in trigrams:
                for cand, prob in trigrams[key].items():
                    cat = (
                        "tri_inc+inc" if kinds.count("<inc>") == 2 else
                        "tri_inc+com" if "<inc>" in kinds else
                        "tri_com+com"
                    )
                    candidates_by_category[cat].append((cand, prob))

    # Bigrams windows: [pos-1,pos], [pos,pos+1]
    left_tok, left_kind = neighbor(pos - 1)
    right_tok, right_kind = neighbor(pos + 1)
    if left_tok is not None:
        for cand, prob in (bigrams.get(left_tok, {}) or {}).items():
            cat = "bi_com" if left_kind == "<com>" else "bi_inc"
            candidates_by_category[cat].append((cand, prob))
    if right_tok is not None:
        # Need inverse bigram index. Approximate by scanning keys where cand followed by right_tok
        for cand_left, nexts in bigrams.items():
            prob = nexts.get(right_tok, None)
            if prob is not None:
                cat = "bi_com" if right_kind == "<com>" else "bi_inc"
                candidates_by_category[cat].append((cand_left, prob))

    ordered: List[str] = []
    ordered.extend(_rank_unique(candidates_by_category["tri_com+com"]))
    ordered.extend([c for c in _rank_unique(candidates_by_category["tri_inc+com"]) if c not in ordered])
    ordered.extend([c for c in _rank_unique(candidates_by_category["tri_inc+inc"]) if c not in ordered])
    ordered.extend([c for c in _rank_unique(candidates_by_category["bi_com"]) if c not in ordered])
    ordered.extend([c for c in _rank_unique(candidates_by_category["bi_inc"]) if c not in ordered])

    # Filter by edit distance for <inc> (maximum_score = 0, as in training)
    if seq[pos][1] == "<inc>":
        pattern = seq[pos][0]
        filtered = [c for c in ordered if min_edit_distance(pattern, c) <= 0.0]

        if include_unigrams_for_inc:
            # Step 7b: Save mappings
            pool = list(unigrams.items())
            if len(pool) > max_unigram_pool:
                # keep top most-probable slice to cap work
                pool = sorted(pool, key=lambda x: -x[1])[:max_unigram_pool]
            extra = [u for u, _ in pool if min_edit_distance(pattern, u) <= 0.0 and u not in filtered]
            # order unigram extras by unigram probability
            extra = [u for u, _ in sorted(((u, unigrams[u]) for u in extra), key=lambda x: -x[1])]
            filtered.extend(extra)

        return filtered

    # For <mis>, we do not have an internal pattern; keep ordered as is
    return ordered


def generate_candidates_for_sequence(
    tagged_seq: TaggedSequence,
    trigrams: TriDict,
    bigrams: BiDict,
    unigrams: UniDict,
    k_cap_per_pos: int,
    use_normalization_for_mis: bool = False,
    trigrams_norm: Optional[TriDict] = None,
    bigrams_norm: Optional[BiDict] = None,
    runic_to_normalized: Optional[Dict[str, str]] = None
) -> CandidatesDict:
    """
    Input:
        tagged_seq: sequence of (token, tag)
        trigrams, bigrams, unigrams: base n-gram models
        use_normalization_for_mis: include normalized bigram/trigram predictions for <mis>
        trigrams_norm, bigrams_norm, runic_to_normalized: normalized resources
        k_cap_per_pos: maximum number of candidates to retain per position
    Output:
        candidates_by_pos: mapping index -> list of candidates (may be empty)
    """
    out: CandidatesDict = {}

    # Helper to get neighbor token only if it's <com>
    def neighbor_token_if_com(idx: int) -> Optional[str]:
        if idx < 0 or idx >= len(tagged_seq):
            return None
        tok, tg = tagged_seq[idx]
        return tok if tg == "<com>" else None

    for i, (_, tag) in enumerate(tagged_seq):
        # Base candidates using the restored old logic (no normalization for <inc>)
        base_cands = generate_candidates_for_position(
            tagged_seq, i, trigrams, bigrams, unigrams, include_unigrams_for_inc=True
        )

        # Enrich <mis> with normalization-based bigram/trigram candidates (tagged)
        if tag == "<mis>" and use_normalization_for_mis and runic_to_normalized and trigrams_norm and bigrams_norm:
            prev_tok = neighbor_token_if_com(i - 1)
            next_tok = neighbor_token_if_com(i + 1)
            nprev = normalize_token_with_map(prev_tok, runic_to_normalized)
            nnext = normalize_token_with_map(next_tok, runic_to_normalized)

            norm_cands: List[Tuple[str, float]] = []

            if nprev in bigrams_norm:
                for cand, prob in bigrams_norm[nprev].items():
                    norm_cands.append((f"{cand} (norm)", prob))

            if nnext:
                for left, nexts in bigrams_norm.items():
                    if nnext in nexts:
                        norm_cands.append((f"{left} (norm)", nexts[nnext]))

            if nprev and nnext and (nprev, nnext) in trigrams_norm:
                for cand, prob in trigrams_norm[(nprev, nnext)].items():
                    norm_cands.append((f"{cand} (norm)", prob))

            norm_ordered = _rank_unique(norm_cands)

            # Merge: keep base order first (old behavior), then normalized (new behavior)
            merged = list(base_cands)
            merged.extend([c for c in norm_ordered if c not in merged])
            out[i] = merged[:max(1, k_cap_per_pos)]
        else:
            out[i] = base_cands[:max(1, k_cap_per_pos)]

    return out


# --------------------------- Scoring and search ---------------------------- #

def backoff_log_prob(
    t_im2: Optional[str],
    t_im1: Optional[str],
    t_i: str,
    unigrams: UniDict,
    bigrams: BiDict,
    trigrams: TriDict,
    epsilon: float = 1e-12,
) -> float:
    """
    Input:
        t_im2: token at i-2 (or None)
        t_im1: token at i-1 (or None)
        t_i: current token
    Output:
        log_p: log probability with simple trigram->bigram->unigram backoff
               (natural log). Uses epsilon floor if all are missing.
    """
    p = None
    if t_im2 is not None and t_im1 is not None:
        p = trigrams.get((t_im2, t_im1), {}).get(t_i, None)
    if p is None and t_im1 is not None:
        p = bigrams.get(t_im1, {}).get(t_i, None)
    if p is None:
        p = unigrams.get(t_i, None)
    if p is None or p <= 0.0:
        p = epsilon
    return math.log(p)


def beam_search_sequences(
    tagged_seq: TaggedSequence,
    candidates: CandidatesDict,
    unigrams: UniDict,
    bigrams: BiDict,
    trigrams: TriDict,
    top_n: int,
    beam_width: int = 50,
    coverage_first: bool = True,
) -> List[Tuple[List[str], int, float]]:
    """
    Input:
        tagged_seq: sequence with <com>/<inc>/<mis>
        candidates: mapping index -> candidate list
        unigrams, bigrams, trigrams: n-gram models
        top_n: how many full-sequence predictions to return
        beam_width: beam size
        coverage_first: prioritize fewer unpredicted positions first
    Output:
        results: list of (sequence_tokens, num_unpredicted, total_logprob) sorted
    """
    # Beam elements: (num_unpredicted, -total_logprob, seq_tokens_so_far)
    beams: List[Tuple[int, float, List[str]]] = [(0, 0.0, [])]

    for i in range(len(tagged_seq)):
        new_beams: List[Tuple[int, float, List[str]]] = []
        cands = candidates.get(i, [])
        if not cands:
            cands = [None]

        for unpred, neg_logp, path in beams:
            prev2 = path[-2] if len(path) >= 2 else None
            prev1 = path[-1] if len(path) >= 1 else None
            for cand in cands:
                if cand is None or cand == "<?>":
                    penalty = 20.0  # strong penalty for missing a position
                    new_beams.append((unpred + 1, neg_logp + penalty, path + ["<?>"]))
                else:
                    logp = backoff_log_prob(prev2, prev1, cand, unigrams, bigrams, trigrams)
                    new_beams.append((unpred, neg_logp - logp, path + [cand]))

        # prune beam
        if coverage_first:
            new_beams.sort(key=lambda x: (x[0], x[1]))
        else:
            new_beams.sort(key=lambda x: x[1])
        beams = new_beams[:beam_width]

    # Final sort and take top_n
    beams.sort(key=lambda x: (x[0], x[1]))
    results = [(seq, unpred, -neg_logp) for (unpred, neg_logp, seq) in beams[:top_n]]
    return results


# --------------------------------- I/O ------------------------------------ #

def read_sequence_from_file_or_prompt() -> List[str]:
    """
    Input:
        None
    Output:
        tokens: list of tokens from 'runic_inscriptions.txt' or user input
    """
    filename = "runic_inscriptions.txt"
    if os.path.exists(filename):
        with open(filename, "r", encoding="utf-8") as f:
            content = f.read().strip()
            if content:
                print(f"Loaded sequence from '{filename}'.")
                return tokenize_text(content)

    user_seq = input("Enter a runic sequence (tokens separated by spaces): ").strip()
    while not user_seq:
        user_seq = input("Please enter a non-empty runic sequence: ").strip()
    return tokenize_text(user_seq)


def pretty_print_mode1(
    original_tokens: List[str],
    tagged_seq: TaggedSequence,
    candidates: CandidatesDict,
    k: int,
) -> None:
    """
    Input:
        original_tokens: original token strings
        tagged_seq: tagged version of original sequence
        candidates: mapping index -> candidate list
        k: maximum candidates per token to display
    Output:
        None (prints a readable summary)
    """
    print("\n=== Per-token predictions ===\n")
    print("Original sequence:")
    print(" ".join(original_tokens))
    print(f"\nPredictions (per token, up to {k} options):\n")
    for i, (tok, tag) in enumerate(tagged_seq):
        cands = candidates.get(i, [])
        if tag == "<com>":
            print(f"Token {i+1}: {tok}  [<com>]")
        else:
            shown = cands[:k]
            bracketed = "[" + ", ".join(shown) + "]" if shown else "[]"
            print(f"Token {i+1}: {tok}  ({tag}) -> {bracketed}")

# ------------------------------- main() ---------------------------------- #

def main() -> None:
    """
    Input:
        None (interactive)
    Output:
        None (prints results to console)
    """
    print("=== Runic N-gram Demo ===")

    # Load base models
    try:
        unigrams, bigrams, trigrams, unigram_list = load_ngram_pickles()
    except FileNotFoundError as e:
        print("Error: Could not load required pickle files.")
        print("Make sure 'unigram_tokens.pkl', 'bigram_tokens.pkl', and 'trigram_tokens.pkl' are in this directory.")
        raise e

    # Ask user whether to include normalization-based predictions
    use_norm = input("\nInclude normalization-based predictions for missing tokens? (y/n): ").strip().lower()
    use_normalization_for_mis = use_norm in {"y", "yes"}

    runic_to_normalized = {}
    unigrams_norm = bigrams_norm = trigrams_norm = None

    if use_normalization_for_mis:
        try:
            runic_to_normalized = load_normalization_mapping("/content/A-Pilot-Study-for-Enhancing-the-Restoration-of-Runic-Inscriptions-Using-Natural-Language-Processing/Notebook/scandi_runic_all_mapping.json")
            unigrams_norm, bigrams_norm, trigrams_norm = load_normalized_ngram_pickles()
            print(f"Loaded normalization mapping and normalized n-gram models ({len(runic_to_normalized)} mappings).")
        except FileNotFoundError:
            print("Warning: Normalization files not found. Continuing without normalization.")
            use_normalization_for_mis = False

    # Get sequence
    tokens = read_sequence_from_file_or_prompt()
    tagged_seq = get_tags(tokens)

    k_str = input("How many options per token (k)? [default: 10] ").strip()
    k = int(k_str) if k_str.isdigit() and int(k_str) > 0 else 10

    candidates = generate_candidates_for_sequence(
        tagged_seq=tagged_seq,
        trigrams=trigrams,
        bigrams=bigrams,
        unigrams=unigrams,
        k_cap_per_pos=max(1, k * 3),
        use_normalization_for_mis=use_normalization_for_mis,
        trigrams_norm=trigrams_norm,
        bigrams_norm=bigrams_norm,
        runic_to_normalized=runic_to_normalized
    )

    pretty_print_mode1(tokens, tagged_seq, candidates, k)

# **Runic N-gram Demo Script**

This script allows you to interactively test the runic n-gram prediction model on your own inscriptions.  
It supports both literal and normalized inference modes, enabling you to explore how normalization affects the model’s predictions.

## Output
The model displays the top-k candidate tokens for each position (including completions for incomplete or missing tokens).

## Optional Normalization
At runtime, you can choose whether to include normalization-based predictions for missing tokens (`(norm)` candidates).  
If enabled, the script loads:
- `scandi_runic_all_mapping.json` (runic → normalized mapping)
- `unigram_tokens_normalized.pkl`, `bigram_tokens_normalized.pkl`, `trigram_tokens_normalized.pkl`

If these files are absent, the script continues in literal mode automatically.

## Recommended Workflow
Because context matters, it is often helpful to **run the model iteratively**:
1. Start with your incomplete or fragmentary inscription.  
2. Run the model once.
3. Integrate the most plausible predicted tokens back into your sequence.  
4. Rerun the model on this updated version to refine predictions for the remaining gaps.

This step-by-step process typically yields more coherent and contextually stable results than a single pass.

## Conventions for the tool
- `-` denotes a **single missing character** within a token.  
- `…` (U+2026) denotes an **unknown or missing sequence**:  
  - As a **standalone token**, it marks a missing token (`<mis>`).  
  - When **inside a token**, it indicates an incomplete token (`<inc>`).

Normalized suggestions derived from the normalized n-gram dictionaries are labeled with `"(norm)"` in the output.


In [ ]:
# @title
if __name__ == "__main__":
    main()